In [1]:
import torch
import torch.nn as nn
import os
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


In [2]:
class CODDataset(Dataset):
    def __init__(self, img_dir):
        self.img_paths = [
            os.path.join(img_dir, f)
            for f in os.listdir(img_dir)
            if f.endswith(('.png', '.jpg'))
        ]

        self.transform = transforms.Compose([
            transforms.Resize((384, 384)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert("RGB")
        return self.transform(img)

In [3]:
from torch.utils.data import Dataset, DataLoader, random_split
dataset = CODDataset(r"CAMO/CAMO_COD_generate_99%/image")

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
print("Train:", len(train_dataset), "Test:", len(test_dataset))

Train: 3200 Test: 800


In [4]:
import sys

sys.path.append(r"C:\Users\TL1\Noisy-COD\code")

from TrainPNet.lib.Network import Network

model = Network()
print("Model created")

C:\Users\TL1\.conda\envs\myexamenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\TL1\.conda\envs\myexamenv\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
C:\Users\TL1\.conda\envs\myexamenv\lib\site-packages\timm\models\registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


pvt v2 b4 loaded!
Model created


In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = Network()

weights = torch.load(r"C:\Users\TL1\Noisy-COD\Net_epoch_best.pth", map_location=DEVICE)

weights_dict = {k.replace('module.', ''): v for k, v in weights.items()}

model.load_state_dict(weights_dict)
model = model.to(DEVICE)
model.eval()

for p in model.parameters():
    p.requires_grad = False

print(" Model ready")

pvt v2 b4 loaded!
 Model ready


In [6]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [7]:
from tqdm import tqdm

pseudo_gts = []

for img in tqdm(dataset, desc="Generating Pseudo GT"):
    img = img.unsqueeze(0).to(DEVICE)

    out = model(img)
    out = out[4] if isinstance(out, (list, tuple)) else out
    out = torch.sigmoid(out).detach().cpu()

    pseudo_gts.append(out)

Generating Pseudo GT:   0%|          | 0/4000 [00:00<?, ?it/s]C:\Users\TL1\Noisy-COD\code\TrainPNet\lib\Network.py:258: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  [f1, F.upsample(f2, size=f1.shape[2:], mode='bilinear', align_corners=True),
C:\Users\TL1\Noisy-COD\code\TrainPNet\lib\Network.py:259: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  F.upsample(f3, size=f1.shape[2:], mode='bilinear', align_corners=True),
C:\Users\TL1\Noisy-COD\code\TrainPNet\lib\Network.py:260: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  F.upsample(f4, size=f1.shape[2:], mode='bilinear', align_corners=True)], 1))
Generating Pseudo GT: 100%|██████████| 4000/4000 [09:00<00:00,  7.40it/s]


In [8]:
noise = torch.zeros((3, 384, 384), requires_grad=True, device=DEVICE)

optimizer = torch.optim.Adam([noise], lr=1e-2)
criterion = nn.BCELoss()

EPOCHS = 3
LAMBDA = 0.01

In [9]:
for epoch in range(EPOCHS):
    total_loss = 0

    for i, imgs in enumerate(train_loader):
        imgs = imgs.to(DEVICE)

        adv_imgs = torch.clamp(imgs + noise.unsqueeze(0), 0, 1)

        outputs = model(adv_imgs)
        outputs = outputs[4] if isinstance(outputs, (list, tuple)) else outputs
        outputs = torch.sigmoid(outputs)

        targets = torch.zeros_like(outputs)

        loss = criterion(outputs, targets) + LAMBDA * torch.norm(noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        #  PRINT PROGRESS
        if i % 10 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss {loss.item():.4f}")

    print(f"Epoch {epoch+1} DONE | Avg Loss: {total_loss/len(train_loader):.4f}")

Epoch 1 | Batch 0/800 | Loss 2.8045
Epoch 1 | Batch 10/800 | Loss 3.4812
Epoch 1 | Batch 20/800 | Loss 3.9742
Epoch 1 | Batch 30/800 | Loss 1.5231
Epoch 1 | Batch 40/800 | Loss 0.8366
Epoch 1 | Batch 50/800 | Loss 0.8350
Epoch 1 | Batch 60/800 | Loss 0.8793
Epoch 1 | Batch 70/800 | Loss 0.8966
Epoch 1 | Batch 80/800 | Loss 0.8332
Epoch 1 | Batch 90/800 | Loss 0.8190
Epoch 1 | Batch 100/800 | Loss 0.7608
Epoch 1 | Batch 110/800 | Loss 0.7311
Epoch 1 | Batch 120/800 | Loss 0.6958
Epoch 1 | Batch 130/800 | Loss 0.6849
Epoch 1 | Batch 140/800 | Loss 0.6590
Epoch 1 | Batch 150/800 | Loss 0.5861
Epoch 1 | Batch 160/800 | Loss 0.6293
Epoch 1 | Batch 170/800 | Loss 0.5480
Epoch 1 | Batch 180/800 | Loss 0.5146
Epoch 1 | Batch 190/800 | Loss 0.5010
Epoch 1 | Batch 200/800 | Loss 0.4968
Epoch 1 | Batch 210/800 | Loss 0.5151
Epoch 1 | Batch 220/800 | Loss 0.4787
Epoch 1 | Batch 230/800 | Loss 0.4670
Epoch 1 | Batch 240/800 | Loss 0.4489
Epoch 1 | Batch 250/800 | Loss 0.4585
Epoch 1 | Batch 260/800

In [10]:
def MAE(pred, gt):
    return torch.abs(pred - gt).mean().item()

def F_beta(pred, gt, beta=0.3):
    pred = (pred > 0.5).float()
    tp = (pred * gt).sum()
    precision = tp / (pred.sum() + 1e-8)
    recall = tp / (gt.sum() + 1e-8)
    return ((1+beta**2)*precision*recall/(beta**2*precision+recall+1e-8)).item()

def E_measure(pred, gt):
    pred = pred - pred.mean()
    gt = gt - gt.mean()
    return (2*(pred*gt)/(pred**2 + gt**2 + 1e-8)).mean().item()

def S_measure(pred, gt):
    return 1 - MAE(pred, gt)

In [11]:
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [14]:
class PseudoDataset(Dataset):
    def __init__(self, dataset, pseudo_gts):
        self.dataset = dataset
        self.pseudo_gts = pseudo_gts

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx], self.pseudo_gts[idx]

In [17]:
def MAE(pred, gt):
    return torch.abs(pred - gt).mean().item()

def F_beta(pred, gt, beta=0.3):
    pred = (pred > 0.5).float()
    tp = (pred * gt).sum()
    precision = tp / (pred.sum() + 1e-8)
    recall = tp / (gt.sum() + 1e-8)
    return ((1+beta**2)*precision*recall/(beta**2*precision+recall+1e-8)).item()

def E_measure(pred, gt):
    pred = pred - pred.mean()
    gt = gt - gt.mean()
    return (2*(pred*gt)/(pred**2 + gt**2 + 1e-8)).mean().item()

def S_measure(pred, gt):
    return 1 - MAE(pred, gt)

In [19]:
metrics_clean = {"MAE": [], "F": [], "E": [], "S": []}
metrics_attack = {"MAE": [], "F": [], "E": [], "S": []}

for idx in test_dataset.indices:   

    img = dataset[idx].unsqueeze(0).to(DEVICE)
    gt  = pseudo_gts[idx].to(DEVICE)

    # Clean
    clean = model(img)
    clean = clean[4] if isinstance(clean, (list, tuple)) else clean
    clean = torch.sigmoid(clean)

    # Attack
    adv_img = torch.clamp(img + noise.unsqueeze(0), 0, 1)
    adv = model(adv_img)
    adv = adv[4] if isinstance(adv, (list, tuple)) else adv
    adv = torch.sigmoid(adv)

    # Metrics
    metrics_clean["MAE"].append(MAE(clean, gt))
    metrics_clean["F"].append(F_beta(clean, gt))
    metrics_clean["E"].append(E_measure(clean, gt))
    metrics_clean["S"].append(S_measure(clean, gt))

    metrics_attack["MAE"].append(MAE(adv, gt))
    metrics_attack["F"].append(F_beta(adv, gt))
    metrics_attack["E"].append(E_measure(adv, gt))
    metrics_attack["S"].append(S_measure(adv, gt))

In [20]:
import numpy as npdef avg(d):
    return {k: np.mean(v) for k, v in d.items()}

clean_avg = avg(metrics_clean)
attack_avg = avg(metrics_attack)

df = pd.DataFrame({
    "Metric": ["MAE ↓", "Fβ ↑", "Eθ ↑", "Sα ↑"],
    "Clean": [
        clean_avg["MAE"],
        clean_avg["F"],
        clean_avg["E"],
        clean_avg["S"]
    ],
    "Attack": [
        attack_avg["MAE"],
        attack_avg["F"],
        attack_avg["E"],
        attack_avg["S"]
    ]
})

print(df)

NameError: name 'np' is not defined

In [ ]:
def avg(d):
    return {k: np.mean(v) for k, v in d.items()}

clean_avg = avg(metrics_clean)
attack_avg = avg(metrics_attack)

df = pd.DataFrame({
    "Metric": ["MAE ↓", "Fβ ↑", "Eθ ↑", "Sα ↑"],
    "Clean": [
        clean_avg["MAE"],
        clean_avg["F"],
        clean_avg["E"],
        clean_avg["S"]
    ],
    "Attack": [
        attack_avg["MAE"],
        attack_avg["F"],
        attack_avg["E"],
        attack_avg["S"]
    ]
})

print(df)

In [ ]:
from tqdm.notebook import tqdm

pseudo_gts = []

for img in tqdm(dataset, desc="Generating Pseudo GT"):
    img = img.unsqueeze(0).to(DEVICE)

    out = model(img)
    out = out[4] if isinstance(out, (list, tuple)) else out
    out = torch.sigmoid(out).detach().cpu()

    pseudo_gts.append(out)